# Stable Diffusion-Based Image Generation Web Application Using Fast API & React

Run from a fresh Python 3.11 kernel. Dependencies install in this kernel. Application files are checksum-verified from a local clone or the companion repository. All writes use an owned temporary workspace. No API keys required. Runs the full original 50-step, 256×256 Stable Diffusion v1-4 example with the safety checker enabled. CUDA uses float16; Apple MPS and CPU use float32. Node 22.12+ is needed for the frontend. 

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,importlib.util,socket,threading,time
from pathlib import Path
from tempfile import TemporaryDirectory
from io import BytesIO
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check', 'torch==2.10.0','torchvision==0.25.0','diffusers==0.36.0','transformers==4.57.6','accelerate==1.12.0','fastapi==0.115.12','uvicorn==0.34.2','httpx==0.28.1','numpy==2.3.5','pillow==12.0.0','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
import numpy as np
import torch
from PIL import Image
from IPython.display import display
torch.set_num_threads(2)
workspace=TemporaryDirectory(prefix='tdb-coding09-')
work=Path(workspace.name)
files={'apps/stable-diffusion-web/api.py': 'd4b31c611829a66df93efb7b0d9bb213dba13580471e9e3e12a32900a697627f', 'apps/stable-diffusion-web/requirements.txt': '5c4e42657b5c74540c5b3da491921b0590ff29e70e738ea1d732bfa627f75dae', 'apps/stable-diffusion-web/client/package.json': 'e2c66232b11866228215b1069071203cc9efc687cf42005149e5bf5808982c12', 'apps/stable-diffusion-web/client/package-lock.json': '6bfdfe4b0184bbf965139fd959744fd5dc7ab713cb42d8c4e87580ddbe854554', 'apps/stable-diffusion-web/client/index.html': 'c8e640449f356cbbacdde7d60e2e2490d58d7298cbf5fb21496d198d2462d56c', 'apps/stable-diffusion-web/client/src/main.jsx': 'd079027bc51e08558bd11b70fb19a1913cba8fa93bda7a2b5f2bf66eea483abe', 'apps/stable-diffusion-web/client/src/App.jsx': 'f0183ca874c7d155df3e95c64290276ac11ebc07255d8af38eb6c4e012c4ebc4', 'apps/stable-diffusion-web/client/src/App.test.jsx': 'ea3f5bd1c9f963d6d444431fe6472a8e1aa9993ac3e213300cf65cae89a9b3e7'}
local_candidates=[p for p in [Path.cwd(),*Path.cwd().parents] if (p/'README.md').exists()]
local_repo=next((p for p in local_candidates if (p/next(iter(files))).exists()),None)
for relative,digest in files.items():
    target=work/relative;target.parent.mkdir(parents=True,exist_ok=True)
    if local_repo is not None:
        payload=(local_repo/relative).read_bytes()
    else:
        url='https://raw.githubusercontent.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/main/'+relative
        with urllib.request.urlopen(url,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,relative
    target.write_bytes(payload)
print('Python:',sys.version.split()[0],'| PyTorch:',torch.__version__)
print('Pinned dependencies and checksum-verified application files ready.')
app_dir=work/'apps/stable-diffusion-web'
client_dir=app_dir/'client'
print('CUDA:',torch.cuda.is_available(),'| Apple MPS:',torch.backends.mps.is_available())

In [ ]:
subprocess.run(['git','clone','--quiet','https://github.com/youssefHosni/Stable-Diffusion-Crash-Course.git',str(work/'original')],check=True)
subprocess.run(['git','checkout','--quiet','e26a01d6a71e23948639aac19d6ee668c2eedccc'],cwd=work/'original',check=True)
print('Cloned original source at immutable revision e26a01d6a71e23948639aac19d6ee668c2eedccc.')

In [ ]:
import venv
venv.EnvBuilder(with_pip=True,symlinks=os.name!="nt").create(work/'stable_diffusion_app')
env_python=work/'stable_diffusion_app'/('Scripts/python.exe'if os.name=='nt'else'bin/python')
print('Created stable_diffusion_app using Python',sys.version.split()[0])

In [ ]:
assert env_python.exists()
print('Use the environment interpreter explicitly; no notebook shell activation required.')

In [ ]:
version=subprocess.run([str(env_python),'-m','pip','--version'],check=True,capture_output=True,text=True)
assert 'pip'in version.stdout
print('Built-in venv includes pip; no virtualenv package required.')

In [ ]:
assert(work/'stable_diffusion_app'/'pyvenv.cfg').exists()
print('The previously created virtual environment is ready.')

In [ ]:
result=subprocess.run([str(env_python),'-c','import sys;print(sys.prefix != sys.base_prefix)'],check=True,capture_output=True,text=True)
assert result.stdout.strip()=='True'
print('Explicit interpreter runs inside the isolated environment.')

In [ ]:
assert sys.prefix!=str(work/'stable_diffusion_app')
print('The notebook kernel remains unchanged; no shell deactivation needed.')

In [ ]:
import fastapi,diffusers,transformers
print('FastAPI:',fastapi.__version__,'| Diffusers:',diffusers.__version__,'| Transformers:',transformers.__version__)

In [ ]:
spec=importlib.util.spec_from_file_location('sd_api',app_dir/'api.py')
api=importlib.util.module_from_spec(spec);spec.loader.exec_module(api)
print('FastAPI, Response, CORSMiddleware, PyTorch and Diffusers imported.')

In [ ]:
assert api.app.title=='FastAPI'
print('FastAPI application instance created.')

In [ ]:
cors=api.app.user_middleware[0].kwargs
assert cors['allow_credentials']is False and '*'not in cors['allow_origins']
print('CORS allows the two local React origins without credentialed wildcards.')

In [ ]:
pipe=api.get_pipeline()
assert pipe.safety_checker is not None
print('Loaded:',api.MODEL_ID,'| device:',api.DEVICE,'| dtype:',api.DTYPE)
print('Original checkpoint pinned; safety checker enabled.')

In [ ]:
prompt='A photorealistic image of an astronaut riding a horse on mars'
direct=api.generate(prompt,1024)
direct_png=direct.body
image=Image.open(BytesIO(direct_png));image.load()
assert image.size==(256,256)and image.format=='PNG'
display(image)
print('Complete 50-step generation:',image.size,'| PNG response bytes:',len(direct_png))

In [ ]:
# The standalone backend is the same complete module used above.
api_source=(app_dir/'api.py').read_text();compile(api_source,'api.py','exec')
assert not(app_dir/'testimage.png').exists()
print('Complete standalone api.py compiled; PNG response stays in memory.')

In [ ]:
import uvicorn,httpx
sock=socket.socket();sock.bind(('127.0.0.1',0));sock.listen(128)
port=sock.getsockname()[1]
server=uvicorn.Server(uvicorn.Config(api.app,log_level='warning'))
thread=threading.Thread(target=server.run,kwargs={'sockets':[sock]},daemon=True);thread.start()
for attempt in range(100):
    if server.started:break
    time.sleep(.1)
assert server.started
try:
    with httpx.Client(base_url=f'http://127.0.0.1:{port}',timeout=600)as client:
        assert client.get('/docs').status_code==200
        response=client.get('/',params={'prompt':prompt,'seed':1024},headers={'Origin':'http://localhost:5173'})
        response.raise_for_status()
        assert response.headers['content-type']=='image/png'
        assert response.content==direct_png
        assert response.headers['access-control-allow-origin']=='http://localhost:5173'
    print('Real Uvicorn HTTP request returns a byte-identical seeded PNG; docs and CORS passed.')
finally:
    server.should_exit=True;thread.join(30);sock.close()
    assert not thread.is_alive()
    print('Owned Uvicorn server stopped.')

In [ ]:
client_source=(client_dir/'src/App.jsx').read_text()
assert '@chakra-ui/react'in client_source and 'axios'in client_source
print('React state, Chakra UI and Axios client source ready.')

In [ ]:
assert 'responseType: "blob"'in client_source
assert 'params: { prompt: prompt.trim(), seed: 1024 }'in client_source
print('Axios sends encoded query parameters and receives an image blob.')

In [ ]:
assert 'finally'in client_source and 'URL.revokeObjectURL'in client_source
print('Client includes loading/error recovery and image object-URL cleanup.')

In [ ]:
subprocess.run(['npm','ci','--no-audit','--no-fund'],cwd=client_dir,check=True)
subprocess.run(['npm','run','build'],cwd=client_dir,check=True)
subprocess.run(['npm','test'],cwd=client_dir,check=True)
print('Pinned React frontend production build and all DOM interaction tests passed.')

In [ ]:
from fastapi.testclient import TestClient
with TestClient(api.app)as client:
    for params in [{},{'prompt':''},{'prompt':'  '},{'prompt':'x'*501},{'prompt':'horse','seed':-1},{'prompt':'horse','seed':2**32}]:
        assert client.get('/',params=params).status_code==422
    ok=client.options('/',headers={'Origin':'http://localhost:5173','Access-Control-Request-Method':'GET'})
    assert ok.status_code==200 and ok.headers['access-control-allow-origin']=='http://localhost:5173'
    bad=client.options('/',headers={'Origin':'https://untrusted.invalid','Access-Control-Request-Method':'GET'})
    assert bad.status_code==400 and 'access-control-allow-origin'not in bad.headers
assert pipe.safety_checker is not None
assert image.size==(256,256)
assert not(app_dir/'testimage.png').exists()
print('Blank/absent/oversized prompts, seed bounds, allowed/rejected origins and safety-check retention passed.')
print('Actual direct inference and real HTTP response matched exactly with the full original 50 steps.')
workspace.cleanup()
print('Owned temporary workspace cleaned up.')

## Original source code

These blocks are preserved verbatim for comparison, not executed.

### Original block 1

```text
gh repo clone youssefHosni/Stable-Diffusion-Crash-Course
```

### Original block 2

```text
conda create --name stable_diffusion_app python=3.8
```

### Original block 3

```text
conda activate stable_diffusion_app
```

### Original block 4

```text
pip install virtualenv
```

### Original block 5

```text
virtualenv stable_diffusion_app
```

### Original block 6

```text
> myenv\Scripts\activate
```

### Original block 7

```text
$ deactivate
```

### Original block 8

```text
pip install -r requirements.txt
```

### Original block 9

```text
from auth_token import auth_token
from fastapi import FastAPI, Response
from fastapi.middleware.cors import CORSMiddleware
import torch
print(torch.cuda.is_available())
from torch import autocast
from diffusers import StableDiffusionPipeline
from io import BytesIO
import base64
```

### Original block 10

```text
app = FastAPI()
```

### Original block 11

```text
app.add_middleware(
    CORSMiddleware, 
    allow_credentials=True, 
    allow_origins=["*"], 
    allow_methods=["*"], 
    allow_headers=["*"]
)
```

### Original block 12

```text
device = torch.device('cuda')
# model_id = "stabilityai/stable-diffusion-2-1"
model_id = "CompVis/stable-diffusion-v1-4"
pipe = StableDiffusionPipeline.from_pretrained(model_id, revision="fp16", torch_dtype=torch.float16, use_auth_token=auth_token) #load the stable diffusion pipeline 
pipe.safety_checker = lambda images, clip_input: (images, False) # to stop the deteecting NSFW
pipe.enable_attention_slicing()
pipe.to(device)
```

### Original block 13

```text
# define Fast API endpoint that generate an image based on a given prompt string. 

@app.get("/")
def generate(prompt: str): 
 
    image = pipe(prompt, 
                 guidance_scale=7,  # Forces generation to better match the prompt, 7 or 8.5 give good results, results are better the larger the number is, but will be less diverse
                 height=256, 
                 width=256).images[0]

    image.save("testimage.png") # save the image

    # return the image
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    imgstr = base64.b64encode(buffer.getvalue())
    return Response(content=imgstr, media_type="image/png")
```

### Original block 14

```text
from auth_token import auth_token
from fastapi import FastAPI, Response
from fastapi.middleware.cors import CORSMiddleware
import torch
print(torch.cuda.is_available())
from torch import autocast
from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
from io import BytesIO
import base64 


# create the Fast API instance and set the configurations 
app = FastAPI()

app.add_middleware(
    CORSMiddleware, 
    allow_credentials=True, 
    allow_origins=["*"], 
    allow_methods=["*"], 
    allow_headers=["*"]
)


device = torch.device('cuda')
# model_id = "stabilityai/stable-diffusion-2-1"
model_id = "CompVis/stable-diffusion-v1-4"
pipe = StableDiffusionPipeline.from_pretrained(model_id, revision="fp16", torch_dtype=torch.float16, use_auth_token=auth_token) #load the stable diffusion pipeline 
pipe.safety_checker = lambda images, clip_input: (images, False) # to stop the deteecting NSFW
pipe.enable_attention_slicing()
pipe.to(device)

@app.get("/")
def generate(prompt: str): 
 
    image = pipe(prompt, 
                 guidance_scale=7,  # Forces generation to better match the prompt, 7 or 8.5 give good results, results are better the larger the number is, but will be less diverse
                 height=256, 
                 width=256).images[0]

    image.save("testimage.png") # save the image

    # return the image
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    imgstr = base64.b64encode(buffer.getvalue())
    return Response(content=imgstr, media_type="image/png")
```

### Original block 15

```text
uvicorn api:app --reload
```

### Original block 16

```text
import {
  ChakraProvider,
  Heading,
  Container,
  Text,
  Input,
  Button,
  Wrap,
  Stack, 
  Image,
  Link,
  SkeletonCircle,
  SkeletonText,
} from "@chakra-ui/react";
import axios from "axios";
import { useState } from "react";
```

### Original block 17

```text
const App = () => {
  const [image, updateImage] = useState();
  const [prompt, updatePrompt] = useState();
  const [loading, updateLoading] = useState();

  const generate = async (prompt) => {
    updateLoading(true);
    const result = await axios.get(`http://127.0.0.1:8000/?prompt=${prompt}`);
    updateImage(result.data);
    updateLoading(false);
  };
```

### Original block 18

```text
return (
    <ChakraProvider>
      <Container>
        <Heading>Stable DIffusion🚀</Heading>
        <Text marginBottom={"10px"}>
          This react application leverages the model trained by Stability AI and
          Runway ML to generate images using the Stable Diffusion Deep Learning
          model. The model can be found via github here{" "}
          <Link href={"https://github.com/CompVis/stable-diffusion"}>
            Github Repo
          </Link>
        </Text>

        <Wrap marginBottom={"10px"}>
          <Input
            value={prompt}
            onChange={(e) => updatePrompt(e.target.value)}
            width={"350px"}
          ></Input>
          <Button onClick={(e) => generate(prompt)} colorScheme={"yellow"}>
            Generate
          </Button>
        </Wrap>

        {loading ? (
          <Stack>
            <SkeletonCircle />
            <SkeletonText />
          </Stack>
        ) : image ? (
          <Image src={`data:image/png;base64,${image}`} boxShadow="lg" />
        ) : null}
      </Container>
    </ChakraProvider>
  );
};
```

### Original block 19

```text
npm install react-scripts --save
```